# NLP Multi-Label Emotion Logistic Regression Classifier

* Building a simple insight sentiment classifier.

* We will use the GoEmotions datset from Google.
  - https://github.com/google-research/google-research/tree/master/goemotions?utm_source=chatgpt.com

* GoEmotions is a corpus of 58k carefully curated comments extracted from Reddit, with human annotations to 27 emotion categories or Neutral.
  - Number of examples: 58,009.

  - Number of labels: 27 + Neutral.

  - Maximum sequence length in training and evaluation datasets: 30.

* On top of the raw data, we also include a version filtered based on reter-agreement, which contains a train/test/validation split:
  - Size of training dataset: 43,410.

  - Size of test dataset: 5,427.

  - Size of validation dataset: 5,426.

* The emotion categories are: admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire, disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief, joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise.


In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

## Loading in the training data.

* Using the official preprocessed GoEmotions training split.

* Each text example is associated with one or more emotion labels.

* For this project, we will create a MULTI-label classification task.

In [2]:
# Load the training data.
train_df = pd.read_csv(
    "data/preprocessed/train.tsv",
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"]
)

train_df.head()

,text,emotion_ids,id
0,My favourite food is anything I didn't have to...,27,eebbqej
1,"Now if he does off himself, everyone will thin...",27,ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj
3,To make her feel threatened,14,ed7ypvh
4,Dirty Southern Wankers,3,ed0bdzj


In [3]:
print(train_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [4]:
print(train_df.shape)

(43410, 3)


In [5]:
# Load the official dev split (used as validation for now)
val_df = pd.read_csv(
    "data/preprocessed/dev.tsv",
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"]
)

In [6]:
val_df.head()

,text,emotion_ids,id
0,Is this in New Orleans?? I really feel like th...,27,edgurhb
1,"You know the answer man, you are programmed to...","4,27",ee84bjg
2,I've never been this sad in my life!,25,edcu99z
3,The economy is heavily controlled and subsidiz...,"4,27",edc32e2
4,He could have easily taken a real camera from ...,20,eepig6r


In [7]:
print(val_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [8]:
print(val_df.shape)

(5426, 3)


In [9]:
# Load in test.
test_df = pd.read_csv(
    "data/preprocessed/test.tsv",
    sep="\t",
    header=None,
    names=["text", "emotion_ids", "id"]
)

In [10]:
print(test_df.columns)

Index(['text', 'emotion_ids', 'id'], dtype='str')


In [11]:
print(test_df.shape)

(5427, 3)


## EDA

In [12]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43410 entries, 0 to 43409
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   text         43410 non-null  str  
 1   emotion_ids  43410 non-null  str  
 2   id           43410 non-null  str  
dtypes: str(3)
memory usage: 4.2 MB


* We have 43,140 rows with no missing or null values.

* The data type for all rows are strings.

In [13]:
# Most common emotion ID in the training set?
train_df['emotion_ids'].value_counts()

emotion_ids
27            12823
0              2710
4              1873
15             1857
1              1652
              ...  
0,12,13,26        1
1,2,5,17          1
13,14             1
3,9,12            1
0,1,18            1
Name: count, Length: 711, dtype: int64

In [14]:
# 27 is the most common id with a count of 12823.

# Some class imblance.

train_df['emotion_ids'].value_counts().head(20)

emotion_ids
27    12823
0      2710
4      1873
15     1857
1      1652
3      1451
18     1427
10     1402
7      1389
2      1025
20      861
6       858
17      853
25      817
26      720
9       709
5       649
22      586
13      510
11      498
Name: count, dtype: int64

* GoEmotions has a fixed mapping from IDs to emotions.
  - admiration
amusement
anger
annoyance
approval
caring
confusion
curiosity
desire
disappointment
disapproval
disgust
embarrassment
excitement
fear
gratitude
grief
joy
love
nervousness
optimism
pride
realization
relief
remorse
sadness
surprise
neutral

In [15]:
train_df['emotion_ids'].nunique()

711

In [16]:
# Some of the dataset is multi-labled, NOT single-labeled.

# Hence why we ended up 711 unique values.
train_df["emotion_ids"].unique()[:30]

<ArrowStringArray>
[    '27',      '2',     '14',      '3',     '26',     '15',   '8,20',
      '0',      '6',    '1,4',      '5',   '3,12',   '6,22', '6,9,27',
     '12',  '16,25',    '2,7',     '17',     '25',   '0,15',  '15,18',
  '16,27',   '7,13',     '10',     '20',      '4',  '13,15',    '0,1',
     '13',      '1']
Length: 30, dtype: str

In [17]:
# Multi-labeled emotion id's contain ",", so let's count how many are multi-labeled.
train_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    36308
True      7102
Name: count, dtype: int64

In [18]:
total_count = len(train_df)
multi_labeled_count = train_df['emotion_ids'].str.contains(',').sum()
single_labeled_count = (~train_df['emotion_ids'].str.contains(',')).sum()

print(f"Total data points: {total_count}")
print(f"Single-labeled data points: {single_labeled_count}")
print(f"Multi-labeled data points: {multi_labeled_count}")

Total data points: 43410
Single-labeled data points: 36308
Multi-labeled data points: 7102


In [19]:
val_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    4548
True      878
Name: count, dtype: int64

In [20]:
test_df['emotion_ids'].str.contains(',').value_counts()

emotion_ids
False    4590
True      837
Name: count, dtype: int64

## Working With Multi-Labeled Data

* How do we want to classifiy a puece of text with MULTIPLE emotions?

* Do we want to find the "dominating" emotion? 
  - Could use probabilistic distributions for this.

* Look into a multi-hot label matrix.
  - Binary vector of 1s and 0s.

  - A FIXED-size numeric-tensor.

  - UNLIKE one-hot encoding, MULTIPLE positions can be 1s at one - hence "multi-hot"

  - EX: (admiration, amusement, love) --> (0, 1, 18) --> (1, 1, 0, ...., 1, ...) where the last '1' is at position 18.

In [21]:
from sklearn.preprocessing import MultiLabelBinarizer


In [22]:
emotion_names = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]

In [23]:
# Extracts column of string of numbers and splits them by commas, and append them to a list.
# EX: ["0, 1, 18"] --> ["0", "1", "18"]
train_df["emotion_id_list"] = (
    train_df["emotion_ids"]
    .str.split(",")
    .apply(lambda ids: [int(i) for i in ids])
)

In [24]:
train_df.tail()

,text,emotion_ids,id,emotion_id_list
43405,Added you mate well I’ve just got the bow and ...,18,edsb738,[18]
43406,Always thought that was funny but is it a refe...,6,ee7fdou,[6]
43407,What are you talking about? Anything bad that ...,3,efgbhks,[3]
43408,"More like a baptism, with sexy results!",13,ed1naf8,[13]
43409,Enjoy the ride!,17,eecwmbq,[17]


In [25]:
# MultiLabelBinarizer = scikit-learn transformer that converts column of label lists into a multi-hot matrix.
mlb = MultiLabelBinarizer(classes=range(len(emotion_names)))

# "fit" leanrs the label set.
# "transform" converts row's list of IDs into a multi-hot vector.
y = mlb.fit_transform(train_df["emotion_id_list"])

print(y.shape)  # should be (43410, 28)

(43410, 28)


In [26]:
train_df[train_df['emotion_ids'].str.contains(',')]

,text,emotion_ids,id,emotion_id_list
7,We need more boards and to create a bit more s...,"8,20",ef4qmod,"[8, 20]"
11,"Aww... she'll probably come around eventually,...","1,4",edex4ki,"[1, 4]"
15,"Shit, I guess I accidentally bought a Pay-Per-...","3,12",edivtm3,"[3, 12]"
19,Maybe that’s what happened to the great white ...,"6,22",eczq8zg,"[6, 22]"
20,"I never thought it was at the same moment, but...","6,9,27",efdlhs1,"[6, 9, 27]"
...,...,...,...,...
43382,"goat handshake denied Personally, I just thin...","14,27",ef2m53x,"[14, 27]"
43383,it's horrid :/,"14,27",edlbr3j,"[14, 27]"
43388,Fuck these trendy hipster joints. Give me my s...,"2,3",ee3nyiy,"[2, 3]"
43395,Sorry I kind of took it like you were flexing ...,"1,24",eelhhzc,"[1, 24]"


In [27]:
y[:1]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1]])

In [28]:
# Look at the matrix for MULTI-LABELS
y[43382: 43384]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1]])

### Convert validation and test as well.

In [29]:
val_df.head()

,text,emotion_ids,id
0,Is this in New Orleans?? I really feel like th...,27,edgurhb
1,"You know the answer man, you are programmed to...","4,27",ee84bjg
2,I've never been this sad in my life!,25,edcu99z
3,The economy is heavily controlled and subsidiz...,"4,27",edc32e2
4,He could have easily taken a real camera from ...,20,eepig6r


In [30]:
# Same parsing you used for train_df
val_df["emotion_id_list"] = (
    val_df["emotion_ids"]
    .str.split(",")
    .apply(lambda ids: [int(i) for i in ids])
)

In [31]:
val_df.tail()

,text,emotion_ids,id,emotion_id_list
5421,It's pretty dangerous when the state decides w...,14,edyrazk,[14]
5422,I filed for divorce this morning. Hoping he mo...,20,edi2z3y,[20]
5423,"The last time it happened I just said, ""No"" an...",10,eewbqtx,[10]
5424,I can’t stand this arrogant prick he’s no bett...,3,eefx57m,[3]
5425,::but I like baby bangs:: /tiny voice,18,ed5h3jh,[18]


In [32]:
# transform() NOT fit_transform() — mlb is already fit on train's label set
y_val = mlb.transform(val_df["emotion_id_list"])

In [33]:
print(y_val.shape)

(5426, 28)


In [34]:
test_df["emotion_id_list"] = (
    test_df["emotion_ids"]
    .str.split(",")
    .apply(lambda ids: [int(i) for i in ids])
)

In [35]:
test_df.tail()

,text,emotion_ids,id,emotion_id_list
5422,Thanks. I was diagnosed with BP 1 after the ho...,15,efeeasc,[15]
5423,Well that makes sense.,4,ef9c7s3,[4]
5424,Daddy issues [NAME],27,efbiugo,[27]
5425,So glad I discovered that subreddit a couple m...,0,efbvgp9,[0]
5426,"Had to watch ""Elmo in Grouchland"" one time too...",27,edtjpv6,[27]


In [36]:
print(test_df.shape)

(5427, 4)


In [37]:
# transform() ONLY — mlb and tfidf are already fit on train
y_test = mlb.transform(test_df["emotion_id_list"])

In [38]:
print(y_test.shape)

(5427, 28)


### TF-IDF (Term Frequency and Inverse Document Frequency)

* TF = Term Frequency
  - How often does a word appear in the specific text?

  - EX: "I am really happy and happy today"
    - "happy" appears TWICE. --> leads to relatively higher TF(more importance).

* IDF = Inverse Document Frequency
  - How common is the word across the entire dataset?

  - EX: If "happy" appears in only a SMALL portion of thousands of comments, it's potentially USEFUL for distinguishing emotions.

  - EX: The word "the" appears in almost every comment, it isn't very useful for distinguishing emotions. 
    - TF-IDF then gives it a LOWER weight.

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split

In [40]:
# "Look at individual words and 2-word phrases, but exclude any word or phrase that appears in only ONE training example/document."
tfidf = TfidfVectorizer(
  ngram_range=(1, 2), 
  min_df=2
)

In [41]:
tfidf

,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None


In [42]:
# TF-IDF: fit on train text only,
X_train = tfidf.fit_transform(train_df["text"])

In [43]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 794235 stored elements and shape (43410, 58338)>

In [44]:
# TF-IDF: fit on train text only, transform val text with the same vocab
X_val = tfidf.transform(val_df["text"])

In [45]:
X_test = tfidf.transform(test_df["text"])

## Run Logistic Regression Model

In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import hamming_loss, f1_score, accuracy_score, classification_report


In [47]:
base_lr = LogisticRegression(
    solver="liblinear",
    max_iter=1000,
    class_weight="balanced"
)

# OneVsRestClassifier enables a plain binary classifier (yes/no for ONE thing) into a multi-label classifier that can predict all 28 emotions at once. 
# scikit-learn's implementation of the "Binary Relevance" strategy from the paper.
# clf.predict(X_val) runs all 28 fitted models on each validation example and stacks their yes/no outputs back into a (n_samples, 28) matrix — returning a multi-hot-shaped prediction directly comparable to y_val.
clf = OneVsRestClassifier(base_lr)
clf.fit(X_train, y)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](28,)","[ 0, 1, 2,...,25,26,27]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), LogisticRegre...r='liblinear'), ...]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,28
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,58338
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'


In [51]:
y_pred = clf.predict(X_val)
y_pred[:3]

array([[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 1, 0, 0]])

In [52]:
print("Subset Accuracy:", accuracy_score(y_val, y_pred))
print("Hamming Loss:", hamming_loss(y_val, y_pred))
print("Micro F1:", f1_score(y_val, y_pred, average="micro"))
print("Macro F1:", f1_score(y_val, y_pred, average="macro"))

print(classification_report(y_val, y_pred, target_names=emotion_names, zero_division=0))

Subset Accuracy: 0.2644673792849244
Hamming Loss: 0.05320941498604602
Micro F1: 0.5099418040737148
Macro F1: 0.456249192990485
                precision    recall  f1-score   support

    admiration       0.58      0.80      0.67       488
     amusement       0.67      0.83      0.74       303
         anger       0.35      0.56      0.43       195
     annoyance       0.21      0.44      0.28       303
      approval       0.25      0.43      0.31       397
        caring       0.25      0.58      0.35       153
     confusion       0.22      0.53      0.31       152
     curiosity       0.30      0.58      0.40       248
        desire       0.30      0.62      0.41        77
disappointment       0.19      0.35      0.25       163
   disapproval       0.26      0.53      0.35       292
       disgust       0.30      0.52      0.38        97
 embarrassment       0.49      0.54      0.51        35
    excitement       0.20      0.44      0.28        96
          fear       0.56      0

## Metrics Breakdown

* Standard accuracy metric won't be quite as effective here, as a prediction can't be "partially right".

* Subset Accuracy = labels a prediciton correct ONLY if it matches the entire label set.
  - EX: true = [admiration, love], predicted = [admiration] → WRONG
    (even though it got 1 of 2 right).

  - Subset accuracy will give us a sense of how often the model gets the full range of emotions right for a givne text. Excluding "partial credit". 

  - Subset Accuracy = 0.264. This tells us models gets the full range of emotions correct for a piece of text 26% of the time. We expect low values as this is a multi-label task, and one wrong label or miss on a 3 3-label example means a misclassification.


* Hamming Loss = an error rate that gives PARTIAL CREDIT.
  - Fraction of all label predictions that were WRONG, out of every sample times every 1 of the 28 labels.

  - 0.053 = only ~5.3% of all individual label predictions (43,410 examples * 28 labels each) were wrong.
  LOW is good here (it's a loss/error rate, not a score).


* Micro F1 = pools all TP, FP, FN across ALL 28 labels, then computes ONE F1 score from those totals.
  - As our model is dominated by FREQUENT classes, this sees how well our model does, while being weighed by how common each emotion is.

  - 0.510 — reflects the model's real-world performance if you imagine emotions appearing at their natural
  frequency. A model that only did well on rare emotions but bad on common ones would still score low here.


* Macro F1 = Computes F1 SEPARATELY for each of the 28 labels, then averages those 28 scores with EQUAL weight — regardless of how rare or common each emotion is.
  - Metric that reflects performance on RARE emotions (like grief, relief).
  
  - A model that ignores rare classes gets punished here even if Micro F1 looks fine.

  - 0.456 — lower than Micro F1, meaning the model is noticeably worse on RARE emotions than on common
  ones. 
  
  - Gap between Macro (0.456) and Micro (0.510) = direct measure of "how much does class imbalance impact model?"


* Precision: of all times the model predicted emotion X, how often was it right? 
  - (LOW precision = lots of false positives)

* Recall: of all TRUE occurrences of emotion X, how many did the model catch? 
  - (LOW recall = lots of missed cases)

* F1: harmonic mean of the two — balances both.

* Recall > precision almost everywhere in our results — a direct effect of class_weight="balanced", which pushes the model to predict positive MORE liberally to catch minority classes, at the cost of MORE FALSE POSITIVES.